# 

# Phylogenetic ASVs from 16S V1-V3 and V4 for primer comparison

In [11]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import pearsonr

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table

In [12]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [13]:
# read in gg2 file
db_df = pd.read_csv('../Data/16S/b7c3e691-ea51-4547-94dd-f79f49e41a36/data/2024.09.taxonomy.asv.tsv', sep='\t', usecols=['Feature ID', 'Taxon'])
db_df.head()

,Feature ID,Taxon
0,GB-GCA-003568775.1-MWMI01000008.1,d__Bacteria; p__; c__; o__; f__; g__; s__
1,GB-GCA-001552015.1-CP010514.1,d__Bacteria; p__; c__; o__; f__; g__; s__
2,GB-GCA-000008085.1-AE017199.1,d__Bacteria; p__; c__; o__; f__; g__; s__
3,TAGCCGCACCCCAAGTGGTAGTCATTATTATTGGGCTTAAAGTGTT...,d__Bacteria; p__; c__; o__; f__; g__; s__
4,TACCCGCGCCACGAGTGGTGATCGCGATTATTGGGCCTAAAGGGTT...,d__Bacteria; p__; c__; o__; f__; g__; s__


In [16]:
V1V3_path = '../Data/16S/Tables/174950_rarefied_table_unannotated_absolute_abundances.biom'
V1V3_df = biom_to_df(V1V3_path).T

# V1V3_ASVs = V1V3_df.columns

# Move the index into a column named 'sequence'
V1V3_df = V1V3_df.reset_index().rename(columns={'index': 'Feature ID'})

# Now do the merge

# Now do the merge
merged = V1V3_df.merge(db_df, on='Feature ID', how='left')
merged.head()

,Feature ID,LAMI.RD001.D0.C1,LAMI.RD001.D0.C2,LAMI.RD001.D14.C1,LAMI.RD001.D14.C2,LAMI.RD001.D28.C1,LAMI.RD002.D0.C2,LAMI.RD003.D14.C1,LAMI.RD002.D14.C1,LAMI.RD003.D28.C1,...,LAMI.RD319.D11.C1,LAMI.RD319.D21.C2,LAMI.RD319.D7.C3,LAMI.RD318.D28.C3,LAMI.RD319.D14.C1,LAMI.RD319.D21.C3,LAMI.RD319.D14.C2,LAMI.RD319.D9.C1,LAMI.RD319.D9.C2,Taxon
0,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAG...,2355.0,1781.0,2173.0,1738.0,2352.0,2892.0,3295.0,3434.0,2367.0,...,1299.0,2587.0,972.0,3213.0,1888.0,988.0,2224.0,1219.0,2093.0,NaN
1,ATTGAACGCTGGCGGCATGCTTTACACATGCAAGTCGAACGGCAGC...,0.0,0.0,0.0,0.0,3.0,0.0,1.0,0.0,0.0,...,1628.0,1069.0,2734.0,538.0,1422.0,2709.0,1410.0,2006.0,1411.0,NaN
2,GATGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGAACAG...,83.0,155.0,81.0,137.0,115.0,114.0,85.0,90.0,95.0,...,0.0,10.0,20.0,5.0,8.0,8.0,8.0,13.0,6.0,NaN
3,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAG...,91.0,120.0,253.0,151.0,217.0,13.0,14.0,41.0,36.0,...,14.0,15.0,5.0,8.0,8.0,9.0,27.0,31.0,20.0,NaN
4,GACGAACGCTGGCGGCGTGCCTAATACATGCAAGTAGAACGCTGAA...,77.0,89.0,108.0,225.0,162.0,27.0,3.0,26.0,432.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,NaN


In [19]:
V1V3_ASVs = V1V3_df.columns.to_list
V1V3_ASVs

<bound method IndexOpsMixin.tolist of Index(['Feature ID', 'LAMI.RD001.D0.C1', 'LAMI.RD001.D0.C2',
       'LAMI.RD001.D14.C1', 'LAMI.RD001.D14.C2', 'LAMI.RD001.D28.C1',
       'LAMI.RD002.D0.C2', 'LAMI.RD003.D14.C1', 'LAMI.RD002.D14.C1',
       'LAMI.RD003.D28.C1',
       ...
       'LAMI.RD319.D28.C3', 'LAMI.RD319.D11.C1', 'LAMI.RD319.D21.C2',
       'LAMI.RD319.D7.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD319.D14.C1',
       'LAMI.RD319.D21.C3', 'LAMI.RD319.D14.C2', 'LAMI.RD319.D9.C1',
       'LAMI.RD319.D9.C2'],
      dtype='object', length=237)>

In [6]:
V4_path = '../Data/16S/Tables/174951_rarefied_table_unannotated_absolute_abundances.biom'
V4_df = biom_to_df(V4_path)
V4_ASVs = V4_df.columns
V4_ASVs


Index(['TACGTAGGTGGCAAGCGTTATCCGGAATTATTGGGCGTAAAGCGCGCGTAGGCGGTTTTTTAAGTCTGATGTGAAAGCCCACGGCTCAACCGTGGAGGGTCATTGGAAACTGGAAAACTTGAGTGCAGAAGAGGAAAGTGGAATTCCATG',
       'TACGTAGGGTGCAAGCGTTAATCGGAATTATTGGGCGTAAAGCGAGTGCAGACGGTTACTTAAGCCAGATGTGAAATCCCCAAGCTTAACTTGGGACGTGCATTTGGAACTGGGTGACTAGAGTGTGTCAGAGGGAGGTAGAATTCCACA',
       'TACGTAGGGTGCGAGCGTTGTCCGGAATTACTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCACGTCGTCTGTGAAATCCTAGGGCTTAACCCTGGACGTGCAGGCGATACGGGCTGACTTGAGTACTACAGGGGAGACTGGAATTTCTGG',
       'TACGTAGGGTGCGAGCGTTAATCGGAATTATTGGGCGTAAAGCGAGTGCAGACGGTTGTTTAAGCCAGATGTGAAATCCCCGAGCTTAACTTGGGACGTGCATTTGGAACTGGATAACTAGAGTGTGTCAGAGGGAGGTAGAATTCCACA',
       'TACGTAGGTCCCGAGCGTTGTCCGGATTTATTGGGCGTAAAGCGAGCGCAGGCGGTTAGATAAGTCTGAAGTTAAAGGCTGTGGCTTAACCATAGTACGCTTTGGAAACTGTTTAACTTGAGTGCAAGAGGGGAGAGTGGAATTCCATGT',
       'TACGTAGGGTGCGAGCGTTGTCCGGAATTACTGGGCGTAAAGGGCTCGTAGGTGGTTTGTCGCGTCGTCTGTGAAATTCCGGGGCTTAACTCCGGGCGTGCAGGCGATACGGGCATAACTTGAGTACTGTAGGGGTAACTGGAATTCCTG',
       'TACGTAGGGTGCGAGCGTTGTCCGGA